<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

## **Proyecto Naive Bayes**

**Autora:** Anais Aponte  
**Bootcamp:** 4Geeks Academy – Intro to Machine Learning  
**Proyecto:** Proyecto Naive Bayes  

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### 📝 **Instrucciones**

#### **Análisis de sentimientos**

El objetivo es crear un clasificador de reseñas de la tienda de Google Play.

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### 📝 **Paso 1: Carga del conjunto de datos**

El dataset utilizado en este proyecto es **playstore_reviews.csv**.

Para facilitar la reproducibilidad y evitar dependencias externas, el archivo ha sido previamente descargado y almacenado dentro del repositorio en la siguiente ruta:

📁 `data/raw/playstore_reviews.csv`

De esta forma, el notebook puede ejecutarse directamente sin necesidad de acceder a enlaces externos.

A continuación se realizan primero las importaciones necesarias y, posteriormente, la carga del dataset para iniciar el análisis.

</div>

In [17]:
# Manipulación de datos
import pandas as pd
import numpy as np
# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
# Preprocesamiento
from sklearn.impute import SimpleImputer
# División de datos
from sklearn.model_selection import train_test_split
# Modelos
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
# Métricas
from sklearn.metrics import classification_report
# Importamos la librería para guardar modelos
import joblib
from sklearn.feature_extraction.text import CountVectorizer



In [2]:
# CARGAMOS EL FICHERO CON LOS DATOS A ANALIZAR
df = pd.read_csv('../data/raw/playstore_reviews.csv')

# Visualizamos 10 registros aleatorios del dataset con sample(). Esto nos permite observar una muestra representativa de los datos,
# ya que algunos datasets pueden estar ordenados (por fecha, id, etc.).
df.sample(10)

,package_name,review,polarity
536,com.dropbox.android,very reliable syncing but......the web ui loo...,0
579,com.evernote,"was great, broken with marshmallow i lauded e...",1
477,com.Slack,it sucks! i am always shown offline and unab...,0
715,com.opera.mini.native,"night mode this app is really good actually, ...",0
451,com.whatsapp,its really good but.. not everyone in your ad...,1
844,com.hamropatro,its very helpful,1
240,com.android.chrome,favorite menu update is suboptimal the new up...,0
200,com.supercell.clashofclans,force close it's been 1 or 2 months and y'all...,0
763,com.shirantech.kantipur,improve tapai ko app dheri ramro ra upayogi x...,1
846,com.hamropatro,nice app,1


<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación inicial**

En este dataset se identifican tres variables principales: `package_name`, `review` y `polarity`. 

La variable `review` contiene el texto del comentario y será la principal fuente de información para el modelo, ya que el objetivo es clasificar el sentimiento expresado en dicho texto. Por su parte, `polarity` actúa como variable objetivo, indicando si el comentario es negativo (0) o positivo (1).

Aunque `package_name` forma parte del dataset, no aporta información relevante para la tarea de clasificación de sentimientos, ya que el resultado depende exclusivamente del contenido del texto y no de la aplicación a la que pertenece. Por ello, esta variable será descartada en los siguientes pasos del análisis.

</div>

</div>

<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

A continuación se detallan las variables incluidas en el dataset:

### 📊 Diccionario de Datos

| Variable       | Descripción                                              | Tipo        |
|----------------|----------------------------------------------------------|------------|
| package_name   | Nombre de la aplicación móvil                            | Categórico |
| review         | Comentario o reseña escrita por el usuario               | Texto      |
| polarity       | Variable objetivo (0 = comentario negativo, 1 = positivo)| Numérico   |

Este diccionario de datos será clave para interpretar correctamente las relaciones entre variables durante el análisis exploratorio (EDA).

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2: Estudio de variables y su contenido**

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.1: Inspección inicial del dataset**

</div>

In [3]:
# Obtener las dimensiones
df.shape

(891, 3)

In [4]:
# Obtener información sobre tipos de datos y valores no nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   package_name  891 non-null    str  
 1   review        891 non-null    str  
 2   polarity      891 non-null    int64
dtypes: int64(1), str(2)
memory usage: 21.0 KB


In [5]:
# Valores nulos
df.isnull().sum()

package_name    0
review          0
polarity        0
dtype: int64

In [6]:
# Vemos la proporción y numero de clases (0 y 1) en la variable objetivo
df_classes = df["polarity"].value_counts().to_frame(name="count")
df_classes["proportion"] = df["polarity"].value_counts(normalize=True)

df_classes

,count,proportion
polarity,,
0,584,0.655443
1,307,0.344557


<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación sobre la estructura del dataset**

El dataset contiene 891 registros y 3 variables. Se observa que no existen valores nulos en ninguna de las columnas, por lo que no será necesario realizar tratamiento de datos faltantes.

Las variables `package_name` y `review` son de tipo texto, mientras que `polarity` es numérica y representa la variable objetivo del modelo. 

Se observa un ligero desbalance en la variable objetivo, con mayor proporción de comentarios negativos (0 ≈ 65%) frente a positivos (1 ≈ 35%).


</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.2: Eliminar espacios y convertir a minúsculas el texto**

</div>

In [7]:
# Limpiamos el texto eliminando espacios y convirtiéndolo a minúsculas
df['review'] = df['review'].str.strip().str.lower()
df.sample(10)

,package_name,review,polarity
464,com.whatsapp,my texts aren't delivering. i've checked my in...,0
705,com.opera.mini.native,poor data/wifi transition still doesn't work w...,0
440,com.whatsapp,occasionally i won't receive a notification of...,1
501,com.Slack,slick chat app! reactions are my favorite part...,1
760,com.shirantech.kantipur,good app all is good but sometime its cannot s...,1
195,com.imangi.templerun2,nice game.... its simply amazing...but i would...,1
325,com.viber.voip,on sgh-i727 android 4.1.2 the new platform sho...,0
610,com.evernote,good but ui ease of use still declining i've u...,1
716,com.opera.mini.native,bug fixes required this version is much better...,0
691,com.hamrokeyboard,brilliant but not complete please add more emo...,1


<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.3: Dividir el conjunto de datos en train y test: X_train, X_test, y_train, y_test**

</div>

In [8]:
# Definimos variables predictoras y objetivo (ignorando la variable `package_name` pues no aporta información relevante para la tarea de clasificación de sentimientos)
X = df["review"]
y = df["polarity"]

In [9]:
# Dividimos los datos en train y test manteniendo la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)
X_train.sample(10)

301    wth why i can't login into wechat once i unins...
239    not responding occasionally sometimes chrome j...
742    takes longer time plz make d newspaper downloa...
683    very good apps!!! obviously, no doubt it is ve...
396    my husband and i can't send msgs to each other...
746    worst app ever seen! one of the worst app i ha...
804    slow update. slow. firefox just keeps on getti...
758    hami neoali hamro dukha sukaha ko sathi thanks...
426    chatheads disappears chatheads keep on disappe...
204    hey supercell i would like u to put in a skype...
Name: review, dtype: str

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.4: Transformar el texto en una matriz de recuento de palabras**

</div>

In [11]:
# Convertimos el texto en variables numéricas mediante conteo de palabras
vec_model = CountVectorizer(stop_words="english")

X_train_vec = vec_model.fit_transform(X_train).toarray()
X_test_vec = vec_model.transform(X_test).toarray()

X_train_vec

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(712, 3272))

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 3: Construye un naive bayes (implementacion seleccionada = MultinomialNB)**

</div>

<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Elección del modelo Naive Bayes**

Siguiendo la teoría del módulo, se ha optado por utilizar **MultinomialNB**, ya que este modelo está pensado para trabajar con **conteos o frecuencias**.

En este caso, las variables predictoras han sido generadas mediante CountVectorizer, lo que transforma cada texto en un vector numérico donde cada valor indica cuántas veces aparece una palabra.

Por tanto, como nuestros datos son conteos de palabras, **MultinomialNB es la opción más adecuada** según lo indicado en la teoría.

Se probarán también **GaussianNB** y **BernoulliNB** para comparar resultados, aunque en principio no parecen ser los más apropiados para este tipo de datos.

</div>

In [15]:
# Entrenamos el modelo seleccionado (MultinomialNB)
model_Multi = MultinomialNB()
model_Multi.fit(X_train_vec, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [16]:
# Realizamos predicciones
y_pred_multi = model_Multi.predict(X_test_vec)

In [18]:
# Evaluamos el modelo
print(classification_report(y_test, y_pred_multi))

              precision    recall  f1-score   support

           0       0.84      0.96      0.90       117
           1       0.89      0.66      0.76        62

    accuracy                           0.85       179
   macro avg       0.87      0.81      0.83       179
weighted avg       0.86      0.85      0.85       179



<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">


### 💡 **Observación sobre el rendimiento del modelo con MultinomialNB**

Se observa que el modelo presenta un mejor rendimiento en la detección de la clase negativa (0) que en la positiva (1). Esto se refleja en un recall más elevado para la clase 0 (0.96) frente al de la clase 1 (0.66).<br><br>

Este comportamiento puede estar influenciado por el desbalanceo del dataset, donde predominan los comentarios negativos (aproximadamente 65%) frente a los positivos (35%).<br><br>

Como consecuencia, el modelo tiende a aprender mejor los patrones asociados a la clase mayoritaria, lo que facilita su detección y dificulta la identificación de la clase minoritaria.

</div>


<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 3.1: Construye un naive bayes (GaussianNB)**

</div>

In [ ]:
# Entrenamos el modelo (GaussianNB)
model_Gauss = GaussianNB()
model_Gauss.fit(X_train_vec, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [23]:
# Realizamos predicciones
y_pred_gauss = model_Gauss.predict(X_test_vec)

In [24]:
# Evaluamos el modelo
print(classification_report(y_test, y_pred_gauss))

              precision    recall  f1-score   support

           0       0.84      0.89      0.86       117
           1       0.76      0.68      0.72        62

    accuracy                           0.82       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.81      0.82      0.81       179



<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 3.2: Construye un naive bayes (BernoulliNB)**

</div>

In [25]:
# Entrenamos el modelo (BernoulliNB)
model_Bern = BernoulliNB()
model_Bern.fit(X_train_vec, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"binarize binarize: float or None, default=0.0Threshold for binarizing (mapping to booleans) of sample features.If None, input is presumed to already consist of binary vectors.",0.0
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [26]:
# Realizamos predicciones
y_pred_bern = model_Bern.predict(X_test_vec)

In [27]:
# Evaluamos el modelo
print(classification_report(y_test, y_pred_bern))

              precision    recall  f1-score   support

           0       0.76      0.97      0.85       117
           1       0.87      0.44      0.58        62

    accuracy                           0.78       179
   macro avg       0.82      0.70      0.72       179
weighted avg       0.80      0.78      0.76       179



<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Comparación de modelos Naive Bayes**

Se han evaluado tres implementaciones del algoritmo Naive Bayes: <b>MultinomialNB</b>, <b>GaussianNB</b> y <b>BernoulliNB</b>, observándose diferencias en su rendimiento.

El modelo <b>MultinomialNB</b> obtiene el mejor resultado global, con una accuracy de aproximadamente 0.85, mostrando un equilibrio adecuado entre precision y recall en ambas clases.

Por su parte, <b>GaussianNB</b> presenta un rendimiento ligeramente inferior (accuracy ~0.82), aunque mantiene un comportamiento relativamente equilibrado entre ambas clases.

En cambio, <b>BernoulliNB</b> obtiene el peor resultado (accuracy ~0.78), destacando especialmente su bajo recall en la clase positiva (1), lo que indica que no logra detectar correctamente una parte importante de los comentarios positivos.

Estos resultados confirman lo indicado en la teoría: <b>MultinomialNB es el modelo más adecuado para trabajar con conteos de palabras</b>, como los generados mediante <code>CountVectorizer</code> en este problema de análisis de texto.

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 4: Optimiza el modelo anterior**

</div>

<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación sobre el modelo optimizado**



</div>

------

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 5: Guardar el modelo**

</div>

In [ ]:
# Guardamos el modelo optimizado
joblib.dump(best_model, '../models/decision_tree_diabetes.sav')

['../models/decision_tree_diabetes.sav']